# Journal de bord

Copier-coller les cellules comme canevas pour y remplir les informations.

## 📅 Journal de bord – Jour 1 (09/09/2026)

### 🔧 Étapes / Actions réalisées
- Prise de connaissance de l'ensemble des documents du cas d'usage : mail de cadrage, sujet d'examen (Bank Marketing), canevas `canvas-cas-usage-v2.ipynb`, journal de bord, feuille de route, référentiel de compétences C1-C9.
- Mise en place de l'espace de travail : copie du canevas et du journal comme fichiers de travail, copie du dataset.
- Démarrage de la Phase 1 (cadrage, §1.1-§1.2) : identification du client (banque de détail) et de son problème métier (coût et mauvais ciblage du démarchage téléphonique).
- Réflexion engagée sur le moment où la prédiction doit intervenir dans le processus métier (avant l'appel) — point qui conditionnera le traitement de la variable `duration` plus tard.

### 📝 Observations / Difficultés
- Aucun blocage technique à ce stade, travail encore au niveau du cadrage.
- Point à trancher : formulation précise de l'énoncé du problème IA (§1.2).

### ✅ Prochaines étapes
- Finaliser §1 (bénéficiaires, critères de succès, risques éthiques anticipés).
- Démarrer §2 (identification et dictionnaire des données).

## 📅 Journal de bord – Jour 2 (04/10/2026)

### 🔧 Étapes / Actions réalisées
- Finalisation de la Phase 1 (§1) : bénéficiaires directs/indirects, critères de succès (métier/modèle/opérationnel), risques éthiques anticipés — distinction posée entre variables sensibles pour un usage commercial (âge, job, situation familiale, éducation) et catégories particulières au sens strict de l'article 9 du RGPD (aucune des variables du dataset n'en relève).
- Phase 2 (§2) : description des sources, dictionnaire complet des 21 variables avec repérage des variables sensibles et de `duration` comme fuite d'information potentielle.
- Mise en place du dépôt Git (repo privé dédié), restructuration alignée sur la convention des modules précédents (`notebooks/`, `src/`, `models/`), environnement virtuel (`uv` + `requirements.txt`).
- Phase 3 (§3, EDA) exécutée en entier : qualité des données (12 doublons, `unknown` jusqu'à 20.9 % sur `default`), `pdays` quasi dégénérée (96.3 % à la valeur 999), forte colinéarité entre les indicateurs socio-économiques (`emp.var.rate` / `euribor3m` / `nr.employed` corrélés > 0.9), déséquilibre de la cible confirmé (11.3 % / 88.7 %), relation en U entre âge et taux de souscription, redondance observée entre `job` et `education`.
- Correction de §0.5 (traçabilité) qui était restée au stade de template lors de la première séance.

### 📝 Observations / Difficultés
- Erreur initiale sur la date limite de rendu (confusion entre le 4 octobre indiqué dans le mail et le vendredi réel, le 9 octobre) — à reconfirmer auprès de la formatrice.
- Les constats de l'EDA (redondance `job`/`education`, signal fort de l'âge) viennent étayer concrètement la justification à venir du Scénario 3 (retrait des variables sensibles) plutôt que de rester une hypothèse de principe.

### ✅ Prochaines étapes
- Phase 4 (§4) : split stratifié, pipeline de préparation sans fuite, définition précise des 4 scénarios de variables.
- Phase 5 (§5) : benchmark multi-modèles en validation croisée.

## 📅 Journal de bord – Jour 3 (05/10/2026)

### 🔧 Étapes / Actions réalisées
- Reconsidéré le traitement de `pdays` : la valeur sentinelle 999 avait d'abord été écartée (colonne supprimée), puis recodée en valeur manquante + imputation par la médiane (sur le train) après avoir identifié qu'une suppression pure perdait l'information de récence réelle pour les ~3.7 % de clients déjà recontactés.
- Discussion approfondie sur `job`/`education` : calcul du **Disparate Impact** (règle des 4/5, méthode déjà vue en M2-B1) pour objectiver le risque de discrimination plutôt que de l'affirmer — DI job = 0.219, DI âge = 0.182.
- Clarifié une nuance importante : un écart statistique dans les données historiques (comportement des clients) n'est pas en soi une discrimination commise par la banque ; le risque réel se situe dans l'usage qu'un modèle automatisé en ferait pour décider qui contacter (boucle de rétroaction possible).
- Vérifié avec la formatrice (Discord) la nature de `duration` et la méthode attendue (comparer avec/sans `duration`, expliquer l'écart par le moment où la donnée est connue) — confirmé, conforme à ce qui avait déjà été fait.
- Refait les graphiques de l'EDA (titres, axes, tri, palette) pour qu'ils soient réellement lisibles.
- Phase 5 (§5) : benchmark régression logistique + Random Forest × 4 scénarios, en validation croisée stratifiée (5-fold), puis évaluation finale sur le test set.
- Phase 6 (§6) : tableau comparatif, mesure réelle de la latence d'inférence (200 prédictions chronométrées), recommandation finale.

### 📝 Observations / Difficultés
- Bug trouvé en Phase 6 : les objets modèles (`LogisticRegression`/`RandomForestClassifier`) étaient réutilisés tels quels entre plusieurs scénarios partageant le même type de modèle, ce qui écrasait l'état entraîné du scénario précédent. Les métriques déjà publiées restaient correctes (prédiction effectuée juste après l'entraînement, avant l'écrasement), mais le pipeline stocké pour être réexploité plus tard ne l'était pas. Corrigé avec `sklearn.base.clone()`.
- Résultat principal : retirer `duration` coûte -28 % de F1 (0.642 → 0.462, fuite confirmée) ; retirer en plus les variables sensibles ne coûte quasiment rien (0.462 → 0.464) — c'est ce chiffre qui a tranché l'arbitrage.

### ✅ Prochaines étapes
- Analyse éthique et réglementaire du sujet (RGPD, biais/équité sur les *prédictions* du modèle retenu, pas seulement sur la donnée brute).
- §7 — Interprétation pour la communication client.
- §8.1-8.2 — Persistance du modèle + API basique.